# Implementation of a KANBoost
## Initialisations



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pykan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 3.5 MB/s eta 0:00:00


# KANBoost Imports

In [ ]:
from kan import *
import torch
from sklearn.model_selection import train_test_split

if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

print("****** DEVICE: ",device , " ******\n")

****** DEVICE:  cuda  ******



# KANBoost Preprocessor Class Implementation

In [ ]:
class Preprocessor:
    def __init__(self, page_size, block_size):
        self.page_size = page_size
        self.block_size = block_size

    def ensure_48bit_address(self, load_address):
        # Ensure the load_address is a 48-bit binary string
        return bin(int(load_address, 16))[2:].zfill(48)

    def calculate_delta(self, block1, block2):
        # Calculate the delta between two blocks (binary subtraction)
        return int(block1, 2) - int(block2, 2)

    def split_load_address(self, load_address):
        binary_address = self.ensure_48bit_address(load_address)

        page = binary_address[:self.page_size]  # x Bit (Varies)
        block = binary_address[self.page_size:self.page_size + self.block_size]  # 6 Bit Fixed
        block_offset = binary_address[self.page_size + self.block_size:]  # Remaining bits

        return (page, block, block_offset)


    def unsplit_load_address(self, load_address, block_delta):
        # Convert the load address (hex) to binary
        binary_address = self.ensure_48bit_address(load_address)

        # Split the binary address into page, block, and offset
        current_page = binary_address[:self.page_size]
        current_block = binary_address[self.page_size:self.page_size + self.block_size]
        current_block_offset = binary_address[self.page_size + self.block_size:]
        adjusted_block_int=int(current_block, 2) + block_delta
        if(adjusted_block_int<0):
          adjusted_block_int=0
        # Adjust the block by adding the block delta
        adjusted_block = bin(adjusted_block_int)[2:].zfill(self.block_size)  # Add delta to block and ensure correct bit length
        # Reconstruct the binary address
        reconstructed_binary_address = current_page + adjusted_block + current_block_offset

        # Convert the reconstructed binary address to hexadecimal
        reconstructed_load_address = hex(int(reconstructed_binary_address, 2))[2:].lower()  # Remove '0x' prefix and convert to uppercase

        return reconstructed_load_address

    def preprocess_data(self, data):
        input_features = []
        output_labels = []
        page_blocks = {}
        preprocessed_details = []  # Store detailed information

        for i in range(len(data) - 1):
            instr_id, cycle_count, load_address, instr_ptr, llc_hit_miss = data[i]
            current_page, current_block, current_block_offset = self.split_load_address(load_address)

            _, _, next_load_address, _, _ = data[i+1]
            next_page, next_block, next_block_offset = self.split_load_address(next_load_address)
            # Initialize page_blocks if current_page is not present
            if current_page not in page_blocks:
                page_blocks[current_page] = ['000001']

            # Calculate delta values for the past blocks
            delta1 = delta2 = delta3 = delta4 = delta5 = 1 + 64

            if len(page_blocks[current_page]) > 1:
                delta1 = 64 + self.calculate_delta(page_blocks[current_page][-1], page_blocks[current_page][-2])
            if len(page_blocks[current_page]) > 2:
                delta2 = 64 + self.calculate_delta(page_blocks[current_page][-2], page_blocks[current_page][-3])
            if len(page_blocks[current_page]) > 3:
                delta3 = 64 + self.calculate_delta(page_blocks[current_page][-3], page_blocks[current_page][-4])
            if len(page_blocks[current_page]) > 4:
                delta4 = 64 + self.calculate_delta(page_blocks[current_page][-4], page_blocks[current_page][-5])
            if len(page_blocks[current_page]) > 5:
                delta5 = 64 + self.calculate_delta(page_blocks[current_page][-4], page_blocks[current_page][-5])

            # Calculate delta for the next block (relative to current block)
            next_delta = self.calculate_delta(next_block, current_block)

            # Append input features
            input_features.append((instr_id,load_address, delta1, delta2, delta3, delta4, delta5))

            # Convert next_delta to a 128-dimensional one-hot array and append as the output label
            output_labels.append(next_delta+64)

            # Append the current block to the page's block list
            page_blocks[current_page].append(current_block)

            # Store the details for inspection without delta1, delta2, delta3, and next_delta
            details = {
                "instr_id": instr_id,
                "page": current_page,
                "block": current_block,
                "block_offset": current_block_offset
            }
            preprocessed_details.append(details)

        return input_features, output_labels, preprocessed_details

# Read Data from file Implementation

In [ ]:
# Reading the data from the text file (same as before)
def read_data_from_file(filename):
    data = []
    with open(filename, 'r', newline='', encoding='utf-8') as file:  # Specify newline='' to handle any line endings
        for line in file:
            line = line.rstrip('\n')  # Strip only the trailing newline to preserve leading newlines if necessary
            if line:
                # Split by comma and remove any extra spaces
                fields = [x.strip() for x in line.split(',')]
                instr_id = int(fields[0])
                cycle_count = int(fields[1])
                load_address = fields[2]
                instr_ptr = fields[3]
                llc_hit_miss = int(fields[4])

                # Append as a tuple
                data.append((instr_id, cycle_count, load_address, instr_ptr, llc_hit_miss))
    return data


# Prepare Dataset Function Implementation

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

def prepare_dataset(input_features, output_labels, batch_size=1, device='cpu'):
    """
    Prepares the dataset for training and testing without train-test split, and preserves instruction IDs.

    Args:
        input_features (list or np.array): The input features for the dataset.
        output_labels (list or np.array): The output labels for the dataset.
        batch_size (int): The batch size for data loaders. Default is 1.
        device (str): The device to store the tensors ('cpu' or 'cuda'). Default is 'cpu'.

    Returns:
        dict: A dictionary containing the processed dataset with 'input', 'labels', and 'instr_ids'.
    """
    # Assuming input_features is a list of tuples like (instr_id, feature1, feature2, ..., featureN)
    instr_ids = [x[0] for x in input_features]  # Extracting the instruction IDs
    features = [x[2:] for x in input_features]   # Extracting the actual feature data (excluding instr_id)

    # Convert to PyTorch tensors and move to the specified device
    data_tensor = torch.tensor(features, dtype=torch.float32, device=device)
    target_tensor = torch.tensor(output_labels, dtype=torch.long, device=device)  # Assuming labels are integer values

    # Create data loaders (optional, if you want to batch and shuffle the data)
    data_loader = DataLoader(TensorDataset(data_tensor, target_tensor),
                             batch_size=batch_size, shuffle=True)

    # Initialize tensors for inputs and labels
    all_inputs = torch.empty(0, data_tensor.size(1), device=device)  # Assuming data_tensor has N features
    all_labels = torch.empty(0, dtype=torch.long, device=device)

    # Concatenate all data into a single tensor on the specified device
    for data, labels in data_loader:
        all_inputs = torch.cat((all_inputs, data.to(device)), dim=0)
        all_labels = torch.cat((all_labels, labels.to(device)), dim=0)

    # Return the dataset as a dictionary, including instruction IDs
    dataset = {
        'input': all_inputs,
        'label': all_labels,
        'instr_ids': instr_ids  # Include the instruction IDs in the output (no need to move to device)
    }

    return dataset


  # ------------------------****** Load Dataset for Training only ******  ------------------------
def load_dataset(data,target):
    # Convert to PyTorch tensors
    data_tensor = torch.tensor(data, dtype=torch.float32)
    target_tensor = torch.tensor(target, dtype=torch.long) #This needs to be torch.float32

    # Split dataset into train and test sets
    train_data, test_data, train_target, test_target = train_test_split(data_tensor, target_tensor, test_size=0.2, random_state=20)

    # Create data loaders (optional, if you want to batch and shuffle the data)
    train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(train_data, train_target), batch_size=1, shuffle=False)
    test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(test_data, test_target), batch_size=1, shuffle=False)

    train_inputs = torch.empty(0, 5, device=device)
    train_labels = torch.empty(0, dtype=torch.long,device=device)
    test_inputs = torch.empty(0, 5, device=device)
    test_labels = torch.empty(0,dtype=torch.long,  device=device)

    # Concatenate all data into a single tensor on the specified device
    for data, labels in train_loader:
        train_inputs = torch.cat((train_inputs, data.to(device)), dim=0)
        train_labels = torch.cat((train_labels, labels.to(device)), dim=0)

    for data, labels in test_loader:
        test_inputs = torch.cat((test_inputs, data.to(device)), dim=0)
        test_labels = torch.cat((test_labels, labels.to(device)), dim=0)

    dataset = {}
    dataset['train_input'] = train_inputs
    dataset['test_input'] = test_inputs
    dataset['train_label'] = train_labels
    dataset['test_label'] = test_labels

    return dataset

# KANBoost Entry Point:

In [ ]:
filename = '/content/data_1M.txt'
data = read_data_from_file(filename)

# Initialize the KANBoost Preprocessor
page_size = 36
block_size = 6

preprocessor = Preprocessor(page_size, block_size)

# Use the preprocess_data function to process the entire data and capture details
input_features, output_labels, preprocessed_details = preprocessor.preprocess_data(data)
# dataset=prepare_dataset(input_features=input_features,output_labels=output_labels,device=device)

In [ ]:
input_features[0]

(1314408, '28e837c883c0', 65, 65, 65, 65, 65)

In [ ]:
last_5_elements = [t[-5:] for t in input_features]

In [ ]:
dataset=load_dataset(last_5_elements,output_labels)

In [ ]:
print("Train data shape: {}".format(dataset['train_input'].shape))
print("Train target shape: {}".format(dataset['train_label'].shape))
print("Test data shape: {}".format(dataset['test_input'].shape))
print("Test target shape: {}".format(dataset['test_label'].shape))


Train data shape: torch.Size([208393, 5])
Train target shape: torch.Size([208393])
Test data shape: torch.Size([52099, 5])
Test target shape: torch.Size([52099])


## Creating and Training the KAN

In [ ]:
# model = KAN(width=[5, 64,128], grid=3, k=5, seed=0, device=device)
model = KAN(width=[5, 64,128], grid=4, k=6, seed=0, device=device)

checkpoint directory created: ./model
saving model version 0.0


In [ ]:
def clear_gpu_memory():
    """Clear GPU memory cache."""
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()  # Wait for all streams on a CUDA device to finish
    print("GPU memory cleared.")
clear_gpu_memory()

GPU memory cleared.


In [ ]:
def Train_KANBoost(dataset):
  def train_acc():
    return torch.mean((torch.argmax(model(dataset['train_input']), dim=1) == dataset['train_label']).float())

  def test_acc():
      return torch.mean((torch.argmax(model(dataset['test_input']), dim=1) == dataset['test_label']).float())

  results = model.fit(dataset, opt="Adam", metrics=(train_acc, test_acc),
                        loss_fn=torch.nn.CrossEntropyLoss(), steps=1000, lamb=0.01, lamb_entropy=8.5, save_fig=False)
  print(results['train_acc'][-1], results['test_acc'][-1])
# 0.025,10.5

In [ ]:

import torch
from torch.utils.data import DataLoader, TensorDataset
def train_model(dataset,num_epochs=1):

  # Step 1: Create TensorDataset for train and test datasets
  train_dataset = TensorDataset(dataset['train_input'], dataset['train_label'])
  test_dataset = TensorDataset(dataset['test_input'], dataset['test_label'])

  # Step 2: Create DataLoader for both train and test datasets
  batch_size = 5000
  train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=False)
  test_loader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False)


  # Step 3: Iterate through the DataLoader for training
  for epoch in range(num_epochs):
      for (train_inputs, train_targets), (test_inputs, test_targets) in zip(train_loader, test_loader):
        partial_dataset = {
          "train_input": train_inputs,
          "train_label": train_targets,
          "test_input": test_inputs,
          "test_label": test_targets
        }
        Train_KANBoost(partial_dataset)

train_model(dataset)

In [ ]:
cp -r /content/model /content/drive/MyDrive/kanboost3_24

# Prefetch File Generation


In [ ]:
dataset=prepare_dataset(input_features=input_features,output_labels=output_labels,device=device)

In [ ]:
dataset['input'][0:5]

tensor([[ 69.,  83.,  68.,  79.,  79.],
        [ 80.,  11.,  57.,  83.,  83.],
        [104.,  74.,  65.,  65.,  65.],
        [ 16.,  82.,  65., 104., 104.],
        [ 78.,  61.,  98.,  28.,  28.]], device='cuda:0')

In [ ]:
model = KAN.loadckpt('/content/drive/MyDrive/kanboost2/'+'0.6')

/usr/local/lib/python3.10/dist-packages/kan/MultKAN.py:571: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(f'{path}_state')
/usr/local/lib/python3.10/dist-

In [ ]:
def generate_prefetch_file(path, prefetches):
    with open(path, 'w') as f:
        for instr_id, pf_addr in prefetches:
            print(instr_id, pf_addr, file=f)

def prefetch_generation(batch_size=5000):
    """
    Prepare dataset with manual batching.
    Returns a list of input-output batches to be processed.
    """
    # Get the total number of samples
    total_samples =len(input_features)

    # Create batches manually by splitting the data
    for i in range(0, total_samples, batch_size):
        batch_inputs = dataset['input'][i:i + batch_size]
        pred=torch.argmax(model(batch_inputs),dim=1)
        for j in range(len(pred)):
          instr_id,load_address,_,_,_,_,_=input_features[i+j]
          load_addr=preprocessor.unsplit_load_address(load_address,int(pred[j].item())-64);
          prefetches.append((instr_id,load_addr))

In [ ]:
prefetches=[]
prefetch_generation()
generate_prefetch_file('prefetch_1M_model_v0.5.txt', prefetches)

In [ ]:
len(prefetches),len(input_features)

(260492, 260492)